# F07: Healthcare Revenue Cycle Management

## What You'll Learn

Revenue Cycle Management (RCM) is the financial backbone of healthcare — it tracks
every dollar from patient encounter through claim submission to final payment. In this
notebook you will:

1. **Generate** a healthcare dataset at medium scale
2. **Explore** the patient, encounter, claim, and claim_line tables
3. **Simulate** a claim processing pipeline with status transitions
4. **Export** SQL INSERT statements for data warehouse loading
5. **Build** a star schema with `dim_patient`, `dim_provider`, and `fact_claim`
6. **Discuss** why HIPAA-safe synthetic data matters for RCM analytics

## Tables We'll Use

| Table | RCM Role |
|---|---|
| `patient` | Demographics, insurance information |
| `encounter` | Office visits, ED, inpatient stays |
| `claim` | Payer claims with status and charge amounts |
| `claim_line` | Line-item detail (CPT codes, units, charges) |
| `provider` | Rendering and billing physicians |
| `facility` | Service locations |

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with healthcare billing concepts (helpful but not required)

## Time Estimate

**~20 minutes**

In [1]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

Installation cell ready. Uncomment the pip line above if needed.


## Step 1 — Generate Healthcare Data at Medium Scale

**What we're about to do:** Generate the full healthcare domain at `medium` scale,
which produces enough data for meaningful RCM analytics — thousands of encounters
and claims.

**Why this matters:** RCM dashboards need volume to show realistic denial rates,
A/R aging distributions, and payer mix patterns. The `medium` scale hits the sweet
spot between speed and analytical richness.

In [2]:
from sqllocks_spindle import Spindle, HealthcareDomain

spindle = Spindle()
data = spindle.generate(domain=HealthcareDomain(), scale="medium", seed=42)

print("Healthcare Domain — Medium Scale")
print("=" * 45)
for table_name, df in data.tables.items():
    print(f"  {table_name:<15} {len(df):>8,} rows")
print(f"\nTotal tables: {len(data.tables)}")
print(f"Total rows:   {sum(len(df) for df in data.tables.values()):,}")

Healthcare Domain — Medium Scale
  facility              50 rows
  patient           25,000 rows
  provider           2,000 rows
  encounter        150,000 rows
  claim            142,500 rows
  diagnosis        270,000 rows
  medication       135,000 rows
  procedure        180,000 rows
  claim_line       356,250 rows

Total tables: 9
Total rows:   1,260,800


## Step 2 — Explore the Core RCM Tables

**What we're about to do:** Examine the patient, encounter, claim, and claim_line
tables — the four tables that form the backbone of any RCM data model.

**Why this matters:** Understanding the shape and relationships of these tables is
essential before building dashboards or ETL pipelines. Each table connects to the
next via foreign keys: patient -> encounter -> claim -> claim_line.

In [3]:
import pandas as pd

# Patient demographics snapshot
patient = data["patient"]
print("=== Patient Table ===")
print(f"Columns: {list(patient.columns)}")
print(f"Rows: {len(patient):,}\n")
print(patient.head(3).to_string(index=False))

# Encounter types
encounter = data["encounter"]
print("\n\n=== Encounter Type Mix ===")
enc_mix = encounter["encounter_type"].value_counts()
enc_pct = encounter["encounter_type"].value_counts(normalize=True).mul(100).round(1)
for val in enc_mix.index:
    print(f"  {val:<15} {enc_mix[val]:>6,}  ({enc_pct[val]}%)")

# Claim status distribution
claim = data["claim"]
print("\n=== Claim Status Distribution ===")
status = claim["status"].value_counts()
status_pct = claim["status"].value_counts(normalize=True).mul(100).round(1)
for val in status.index:
    print(f"  {val:<12} {status[val]:>6,} claims  ({status_pct[val]}%)")

# Claim line detail
claim_line = data["claim_line"]
print(f"\n=== Claim Lines ===")
print(f"Total claim lines: {len(claim_line):,}")
print(f"Avg lines per claim: {len(claim_line) / len(claim):.1f}")
print(claim_line.head(3).to_string(index=False))

=== Patient Table ===
Columns: ['patient_id', 'mrn', 'first_name', 'last_name', 'date_of_birth', 'gender', 'race', 'street', 'city', 'phone', 'email', 'insurance_plan', 'insurance_member_id', 'registration_date', 'is_active', 'state', 'zip_code']
Rows: 25,000

 patient_id         mrn first_name  last_name                 date_of_birth gender     race                       street        city        phone                email             insurance_plan insurance_member_id             registration_date is_active state zip_code
          1 MRN00000001     Ariana Villarreal 1943-02-25 13:12:40.205445379      M    White              2770 Lisa Locks   Delbarton 503-713-6515  hmiller@example.com                  Cigna PPO        INS000000001 2022-03-24 18:34:27.894818304      true    WV    25670
          2 MRN00000002    Cynthia  Hernandez 1939-05-30 19:18:23.460000411      M Hispanic              018 Robert Cape   Taneytown   8205544366                  NaN Blue Cross Blue Shield PPO        

## Step 3 — Claim Processing Pipeline Simulation

**What we're about to do:** Join claims to encounters and patients to simulate an
RCM pipeline view — showing the journey from patient visit through claim adjudication.
We'll compute key RCM metrics: denial rate, average charge, and A/R by status.

**Why this matters:** These are the exact KPIs that RCM directors track daily.
Having realistic synthetic data lets you build and test dashboards without waiting
for production data access or navigating HIPAA compliance reviews.

In [4]:
# Build the RCM pipeline view: claim -> encounter -> patient
# Use claim.patient_id directly (no need to pull from encounter)
pipeline = (
    claim
    .merge(encounter[["encounter_id", "encounter_type"]], on="encounter_id", how="left")
    .merge(patient[["patient_id", "gender", "insurance_plan"]], on="patient_id", how="left")
)

# Key RCM metrics
total_charges = pipeline["total_amount"].sum()
denied = pipeline[pipeline["status"] == "denied"]
denial_rate = len(denied) / len(pipeline) * 100

print("RCM Pipeline Summary")
print("=" * 50)
print(f"Total claims:     {len(pipeline):>10,}")
print(f"Total charges:    ${total_charges:>12,.2f}")
print(f"Denial rate:      {denial_rate:>10.1f}%")
print(f"Avg charge/claim: ${pipeline['total_amount'].mean():>12,.2f}")

# A/R breakdown by status
print("\n=== Accounts Receivable by Status ===")
ar = pipeline.groupby("status")["total_amount"].agg(["count", "sum", "mean"])
ar.columns = ["Claims", "Total Charges", "Avg Charge"]
ar = ar.sort_values("Total Charges", ascending=False)
print(ar.to_string())

# Denial rate by encounter type
print("\n=== Denial Rate by Encounter Type ===")
for enc_type in pipeline["encounter_type"].unique():
    subset = pipeline[pipeline["encounter_type"] == enc_type]
    enc_denials = subset[subset["status"] == "denied"]
    rate = len(enc_denials) / len(subset) * 100 if len(subset) > 0 else 0
    print(f"  {enc_type:<15} {rate:.1f}%  ({len(enc_denials):,} / {len(subset):,})")

RCM Pipeline Summary
Total claims:        142,500
Total charges:    $319,877,300.00
Denial rate:             0.0%
Avg charge/claim: $    2,244.75

=== Accounts Receivable by Status ===
                Claims  Total Charges   Avg Charge
status                                            
Paid            102628    230806530.0  2248.962564
Denied           21681     48266550.0  2226.214197
Pending           6892     15575525.0  2259.942687
Partially Paid    6975     15556205.0  2230.280287
Appealed          4324      9672490.0  2236.931082

=== Denial Rate by Encounter Type ===
  Inpatient       0.0%  (0 / 11,528)
  Outpatient      0.0%  (0 / 100,161)
  Emergency       0.0%  (0 / 14,009)
  Observation     0.0%  (0 / 5,662)
  Telehealth      0.0%  (0 / 11,140)


## Step 4 — Export SQL INSERT Statements for Warehouse Loading

**What we're about to do:** Use `to_sql_inserts()` to generate T-SQL INSERT statements
for loading the healthcare data into a SQL Server, Azure SQL, or Fabric Warehouse.

**Why this matters:** Many RCM systems use SQL databases as their analytical warehouse.
`to_sql_inserts()` generates DDL (CREATE TABLE) plus batched INSERT statements that
you can run directly against any T-SQL compatible endpoint.

In [5]:
from sqllocks_spindle.output import PandasWriter
import os

writer = PandasWriter()
output_dir = "./spindle_healthcare_sql"

sql_files = writer.to_sql_inserts(
    tables=data.tables,
    output_dir=output_dir,
    schema_name="rcm",
    batch_size=500,
    include_ddl=True,
    sql_dialect="tsql",
)

print(f"Generated {len(sql_files)} SQL files in {output_dir}/\n")
for f in sql_files:
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f):<35} {size:>10,} bytes")

# Preview the first few lines of one file
print(f"\n=== Preview: {os.path.basename(sql_files[0])} (first 15 lines) ===")
with open(sql_files[0], "r") as fh:
    for i, line in enumerate(fh):
        if i >= 15:
            print("  ...")
            break
        print(f"  {line.rstrip()}")

Generated 9 SQL files in ./spindle_healthcare_sql/

  facility.sql                             9,411 bytes
  patient.sql                          7,060,807 bytes
  provider.sql                           165,845 bytes
  encounter.sql                       12,484,451 bytes
  claim.sql                           17,257,968 bytes
  diagnosis.sql                       26,103,749 bytes
  medication.sql                       9,814,375 bytes
  procedure.sql                       15,504,078 bytes
  claim_line.sql                      21,789,517 bytes

=== Preview: facility.sql (first 15 lines) ===
  -- Generated by Spindle v2.2.2 (sqllocks-spindle)
  -- Generated: 2026-03-17T22:35:35Z
  
  -- ============================================================
  -- Table: facility (50 rows)
  -- ============================================================
  
  IF OBJECT_ID('[rcm].[facility]', 'U') IS NOT NULL
      DROP TABLE [rcm].[facility];
  GO
  
  CREATE TABLE [rcm].[facility] (
      [facility_id

## Step 5 — Healthcare Star Schema

**What we're about to do:** Transform the 3NF healthcare data into a star schema
with `dim_patient`, `dim_provider`, `dim_facility`, and `fact_claim` (plus the
auto-generated `dim_date`).

**Why this matters:** Star schemas are the foundation of healthcare BI. Power BI,
Fabric DirectLake, and SSAS Tabular all expect dimension/fact models. This transform
gives you a warehouse-ready model from synthetic data in seconds.

In [6]:
from sqllocks_spindle.transform import StarSchemaTransform

transform = StarSchemaTransform()
star = transform.transform(data.tables, HealthcareDomain().star_schema_map())

print(star.summary())

# Show dimensions
print("\n=== Dimension Tables ===")
for name, df in star.dimensions.items():
    print(f"  {name:<25} {len(df):>8,} rows  |  Columns: {len(df.columns)}")

# Show facts
print("\n=== Fact Tables ===")
for name, df in star.facts.items():
    print(f"  {name:<25} {len(df):>8,} rows  |  Columns: {len(df.columns)}")

# Date dimension
print(f"\n=== dim_date ===")
print(f"  Date range: {star.date_dim['date'].min()} to {star.date_dim['date'].max()}")
print(f"  Total days: {len(star.date_dim):,}")

Star Schema Result
DIMENSIONS:
  dim_patient                   25,000 rows  18 cols
  dim_provider                   2,000 rows  8 cols
  dim_facility                      50 rows  12 cols
  dim_date                       1,460 rows  14 cols
FACTS:
  fact_encounter               150,000 rows  11 cols
  fact_claim                   356,250 rows  18 cols

=== Dimension Tables ===
  dim_patient                 25,000 rows  |  Columns: 18
  dim_provider                 2,000 rows  |  Columns: 8
  dim_facility                    50 rows  |  Columns: 12

=== Fact Tables ===
  fact_encounter             150,000 rows  |  Columns: 11
  fact_claim                 356,250 rows  |  Columns: 18

=== dim_date ===
  Date range: 2022-01-02 to 2025-12-31
  Total days: 1,460


## Why HIPAA-Safe Synthetic Data Matters

Working with real Protected Health Information (PHI) for analytics development is
a compliance minefield:

- **Access controls**: PHI requires BAAs, role-based access, audit logging
- **De-identification**: Safe Harbor or Expert Determination methods are expensive
  and error-prone
- **Environment restrictions**: PHI often can't leave production networks, making
  dev/test painful
- **Breach risk**: Every copy of real data is a potential breach vector

**Spindle's healthcare domain solves this** by generating data that:

- Contains **zero real PHI** — every patient, provider, and facility is synthetic
- Follows **realistic clinical distributions** (Census demographics, CMS diagnosis
  frequencies, industry-standard payer mixes)
- Maintains **full referential integrity** — JOINs work exactly as they would on
  production data
- Is **reproducible** — same seed produces identical datasets, enabling regression
  testing

You can share Spindle-generated healthcare data freely across teams, environments,
and cloud regions with no HIPAA compliance burden.

## What's Next?

You've built a complete RCM analytics pipeline from synthetic healthcare data —
from raw encounters through claim processing to a warehouse-ready star schema.

| Notebook | What You'll Learn |
|----------|-------------------|
| **T04: Healthcare Deep Dive** | Detailed exploration of every healthcare table and its distributions |
| **T06: Star Schema Export** | Deep dive into star schema mechanics, Parquet export |
| **T08: Fabric Quickstart** | Load this data directly into a Fabric Lakehouse |
| **[F04 — Real-Time Streaming](./F04_realtime_streaming.ipynb)** | Stream healthcare events in real time with burst windows and anomaly injection |

**Key takeaways:**
- Spindle's healthcare domain generates realistic RCM data at any scale
- Denial rates, payer mixes, and encounter distributions mirror industry benchmarks
- `to_sql_inserts()` gives you T-SQL ready for any SQL Server/Fabric Warehouse
- `StarSchemaTransform` produces a BI-ready model with dim/fact tables
- Synthetic data eliminates HIPAA compliance barriers for dev/test workloads